In [ ]:
import pandas as pd

df = pd.read_csv('../data/learn_dataset.csv')
print(df.head())  # Shows the first 5 rows
print(df.columns) # Shows all column names


In [ ]:
code_Occupation_42 = pd.read_csv('../data/code_Occupation_42.csv')

# Merge on Occupation_42 from df and Code from code_Occupation_42
merged = pd.merge(df, code_Occupation_42, left_on='Occupation_42', right_on='Code', how='left')

print(merged.head())

In [ ]:
from scipy.stats import ks_2samp
import pandas as pd

# Assuming 'merged' is your DataFrame with 'Occupation_42' and 'target' columns
occupations = merged['Occupation_42'].dropna().unique()
ks_results = []

for occupation in occupations:
    target_in_occupation = merged.loc[merged['Occupation_42'] == occupation, 'target']
    target_elsewhere = merged.loc[merged['Occupation_42'] != occupation, 'target']
    if len(target_in_occupation) > 0 and len(target_elsewhere) > 0:
        stat, p_value = ks_2samp(target_in_occupation, target_elsewhere)
        ks_results.append({'Occupation_42': occupation, 'ks_stat': stat, 'p_value': p_value, 'n_in_occupation': len(target_in_occupation)})

# Convert results to DataFrame for easy viewing
ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)

# Show the top 10 departments with the largest KS statistic
# print(ks_df.head(10))

import plotly.express as px

# Sort by KS statistic for better visualization
ks_df_sorted = ks_df.sort_values('ks_stat', ascending=False)

fig = px.bar(
    ks_df_sorted,
    x='Occupation_42',
    y='ks_stat',
    hover_data=['p_value', 'n_in_occupation'],
    labels={'Occupation_42': 'Occupation', 'ks_stat': 'KS Statistic'},
    title='KS Statistic by Occupation (Target Distribution Difference)'
)
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Option 1: Threshold-Based Grouping    
threshold = 0.2  # Adjust as needed
ks_df['group_threshold'] = (ks_df['ks_stat'] >= threshold).astype(int)

# Option 2: Significance-Based Grouping
ks_df['group_significance'] = (ks_df['p_value'] < 0.05).astype(int)

# Prepare subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("KS Statistic Threshold", "Significance (p < 0.05)"))

# Plot for Option 1
fig.add_trace(
    go.Bar(
        x=ks_df['Occupation_42'],
        y=ks_df['ks_stat'],
        marker_color=ks_df['group_threshold'].map({1: 'crimson', 0: 'royalblue'}),
        name='KS Statistic (Threshold)'
    ),
    row=1, col=1
)

# Plot for Option 2
fig.add_trace(
    go.Bar(
        x=ks_df['Occupation_42'],
        y=ks_df['ks_stat'],
        marker_color=ks_df['group_significance'].map({1: 'crimson', 0: 'royalblue'}),
        name='KS Statistic (Significance)'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Occupation Grouping by KS Statistic and Significance',
    xaxis_title='Occupation',
    yaxis_title='KS Statistic',
    showlegend=False,
    width=1100,
    height=500
)

fig.show()

In [ ]:
import plotly.graph_objs as go

# Option 1: Threshold-Based Grouping
threshold = 0.3  # Adjust as needed
ks_df['group'] = ks_df['ks_stat'] >= threshold

# Group 1: Individual departments above threshold
group1 = ks_df[ks_df['group']]
# Group 2: All other departments grouped together
group2 = ks_df[~ks_df['group']]

# For Group 2, aggregate KS stats (e.g., mean or sum)
group2_agg = {
    'Occupation_42': 'Group 2',
    'ks_stat': group2['ks_stat'].mean(),  # or sum, or median
    'count': len(group2)
}

# Prepare data for plotting
x = group1['Occupation_42'].tolist() + [group2_agg['Occupation_42']]
y = group1['ks_stat'].tolist() + [group2_agg['ks_stat']]

# Plot
fig = go.Figure()

# Plot Group 1 departments
fig.add_trace(go.Bar(
    x=group1['Occupation_42'],
    y=group1['ks_stat'],
    name='Group 1 (Above Threshold)',
    marker_color='crimson'
))

# Plot Group 2 as a single bar
fig.add_trace(go.Bar(
    x=[group2_agg['Occupation_42']],
    y=[group2_agg['ks_stat']],
    name=f'Group 2 (Others, n={group2_agg["count"]})',
    marker_color='royalblue'
))

fig.update_layout(
    title='KS Statistic: Group 1 Departments vs Group 2 (Others)',
    xaxis_title='Occupation / Group',
    yaxis_title='KS Statistic',
    width=900,
    height=500,
    showlegend=True
)
fig.show()